# K-1. Import Library dan Inisialisasi

In [1]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Kode diatas bertugas menginstal pustaka faker secara senyap dan mengimpor seluruh pustaka utama seperti numpy, pandas, Faker, dan random yang dibutuhkan untuk memicu pengacakan data serta mengolah data tabular.

# K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

In [3]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} "
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating
    })

df = pd.DataFrame(rows)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris data mentah:", len(df))

Jumlah baris data mentah: 515


Kode diatas ini membuat 500 baris data transaksi sintetis marketplace dengan mengunci SEED = 42 agar hasil pengacakannya konsisten dan dapat direproduksi. Kode ini secara sengaja menyuntikkan berbagai variasi data mentah yang kotor seperti format harga yang bercampur simbol, penulisan tanggal yang tidak seragam, huruf kapital acak, missing value buatan, serta 15 baris duplikat hingga menghasilkan dataset mentah sebanyak 515 baris yang disimpan ke file transaksi_mentah.csv.

# K-3. Deteksi dan Penanganan Missing Value

In [4]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [5]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

Kode ini ngecek berapa banyak data kosong di tiap kolom, dan hasilnya ada 20 nama pelanggan, 16 metode pembayaran, 30 kota, plus 166 rating yang kosong. Baris yang nama pelanggan atau metode pembayarannya kosong langsung dihapus karena info ini wajib ada. Sementara itu, kota yang kosong diisi dengan teks "Tidak Diketahui" biar datanya masih bisa dipakai buat analisis lainnya.

# K-4. Deteksi dan Penanganan Duplicate

In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


Kode diatas melakukan pemeriksaan dan penghapusan baris data duplikat yang tercatat lebih dari sekali akibat potensi gangguan sistem atau jaringan. Perintah duplicated().sum() digunakan untuk menghitung jumlah transaksi yang identik, kemudian drop_duplicates() mengeksekusi penghapusan baris duplikat tersebut sehingga data berkurang 15 baris menjadi 500 baris.

# K-5. Koreksi Tipe Data dan Standardisasi Format

a. Standardisasi teks kategorika

In [7]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

Kode diatas untuk merapikan format teks pada kolom category, payment_method, dan shipping_city. Penggunaan .astype("string") dipadukan dengan .str.strip() dan .str.title() berfungsi menghapus spasi liar di awal atau akhir kata serta menyeragamkan kapitalisasi teks menjadi Title Case tanpa mengubah nilai NaN menjadi string teks nan. Selain itu, perintah penggantian khusus ditambahkan untuk mengembalikan teks "Cod" menjadi kapital penuh COD karena merupakan singkatan singkatan Cash on Delivery.

b. Koreksi tipe data pada kolom price

In [8]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("Rp", "").strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

Kode diatas untuk membersihkan kolom price yang awalnya tersimpan sebagai teks berformat tidak konsisten seperti memiliki spasi, awalan Rp, pemisah ribuan titik, atau akhiran desimal .0. Melalui fungsi bersihkan_harga(), semua karakter non-numerik dan simbol pemisah dihapus secara hati-hati, lalu nilainya dikonversi menjadi tipe numerik desimal (float), sementara data yang kosong atau rusak dikembalikan sebagai

c. Standardisasi format tanggal ke YYYY-MM-DD:

In [9]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

Kode diatas menyamakan seluruh format tanggal yang awalnya tercampur ISO, DD/MM/YYYY, dan DD-MM-YYYY menjadi satu format baku ISO YYYY-MM-DD. Proses parsing dilakukan secara eksplisit menggunakan fungsi parse_tanggal() dengan menguji format satu per satu pada tiap baris data untuk menghindari bug Pandas yang bisa salah membalik posisi tanggal dan bulan jika menggunakan format="mixed" bersama dayfirst=True.

d. Finalisasi tipe data:

In [10]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

Kode diatas melakukan penegasan akhir terhadap tipe data numerik pada Dataframe. Kolom quantity dikunci tipe datanya menjadi bilangan bulat int, sedangkan kolom price dipastikan bertipe bilangan desimal float, sehingga seluruh kolom memiliki tipe data terstruktur yang valid sebelum diekspor ke file CSV.

# K-6. Ekspor Dataset Bersih

In [11]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Kode diatas berfungsi mengekspor dataframe yang sudah bersih dan terstruktur ke dalam file CSV bernama transaksi_bersih.csv tanpa menyertakan indeks baris index=False. Proses ini menghasilkan total 490 baris data bersih yang siap dipakai sebagai input untuk tahap penyimpanan data terstruktur pada praktikum berikutnya.

# STUDI KASUS

# 1)	Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda lakukan.

**jawab**

Angka 515 adalah total baris data mentah yang ditarik oleh tim IT langsung dari sistem. Setelah dilakukan pembersihan data preprocessing, terdapat 25 baris data kotor yang dibuang, yaitu:

-	15 baris duplikat, transaksi sama yang terdeteksi tercatat dua kali, biasanya akibat glitch jaringan saat pembayaran.  
-	10 baris missing value, transaksi cacat yang kehilangan informasi wajib, seperti customer_name atau payment_method.

# 2)	Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep Veracity.

**Jawab**

Ya, 490 baris jauh lebih benar dan valid dibanding 515 baris. Hal ini berkaitan erat dengan dimensi Veracity pada 5V, yang menekankan pada tingkat keakuratan dan keandalan data. Jika tim Finance menggunakan 515 baris, laporan keuangan akan menggelembung secara semu karena menghitung transaksi ganda dan transaksi cacat. Mengacu pada prinsip garbage in, garbage out, pembersihan ini dilakukan agar tim Finance mendapatkan angka yang riil dan dapat dipertanggungjawabkan.


# 3)	Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?

**Jawab**

Kolom rating dibiarkan kosong karena ulasan dari pembeli sifatnya opsional.
Jika missing value dipaksa diisi misalnya di-imputasi dengan nilai rata-rata atau angka 0, tindakan tersebut akan mendistorsi kenyataan dan menimbulkan bias palsu pada data. Untuk kebutuhan tim Finance yang ingin mengetahui rating rata-rata, perhitungan cukup dieksekusi pada pembeli yang memang memberikan ulasan saja secara otomatis mengabaikan NaN di Pandas, tanpa harus merekayasa nilai dari pelanggan yang memilih tidak memberi ulasan.

In [12]:
# Upload file CSV ke Google Drive

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Buat folder khusus (jika belum ada)
folder_path = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(folder_path, exist_ok=True)

# Copy file ke folder tersebut
!cp transaksi_mentah.csv "{folder_path}/"
!cp transaksi_bersih.csv "{folder_path}/"

print("✅ File berhasil diupload ke Google Drive!")
print(f"📁 Lokasi: MyDrive/BigData/Praktikum2")
print(f"📄 File yang diupload:")
print(f"   - transaksi_mentah.csv")
print(f"   - transaksi_bersih.csv")

Mounted at /content/drive
✅ File berhasil diupload ke Google Drive!
📁 Lokasi: MyDrive/BigData/Praktikum2
📄 File yang diupload:
   - transaksi_mentah.csv
   - transaksi_bersih.csv
